# DNAscape smoke tests

This notebook is a lightweight interactive check for the mapping and simulation pipeline. It is not a replacement for formal tests, but it is useful before committing larger changes.


## Setup


In [ ]:
%reload_ext autoreload
%autoreload 2

from dnascape import *

## Minimal pipeline check


In [ ]:
chr_number = 1
output_resolution_kb = 5
interval_kb = (0, 10_000)
fork_speed = 1.4

replication_timing_full = loadcsv("example", example="replication_timing_hESC", chr_number=chr_number)
replication_timing, positions_kb = select_profile_region(
    replication_timing_full,
    output_resolution_kb=output_resolution_kb,
    interval_kb=interval_kb,
)

firing_rate = rmap(
    source={
        "replication_timing": replication_timing,
        "fork_speed": fork_speed,
    },
    target="firing_rate",
    assumptions={"resolution": output_resolution_kb, "maxiter": 2},
)

timing_from_firing = rmap(
    source={
        "firing_rate": firing_rate,
        "fork_speed": fork_speed,
    },
    target="replication_timing",
    assumptions={"resolution": output_resolution_kb},
)

simulation = rsim(
    source={
        "firing_rate": firing_rate,
        "fork_speed": fork_speed,
    },
    assumptions={
        "sim_number": 5,
        "resolution": output_resolution_kb,
        "seed": 1,
        "time_statsQ": True,
        "verbose": False,
    },
)

assert replication_timing.ndim == 1
assert positions_kb.shape == replication_timing.shape
assert firing_rate.shape == replication_timing.shape
assert timing_from_firing.shape == replication_timing.shape
assert simulation["replication_timing"].shape == replication_timing.shape
assert simulation["origin_efficiency"].shape == replication_timing.shape

"DNAscape smoke checks passed"